# Rock energy feature engineering + multiple regime segmentation + Plotly surfaces

Вход: `united.csv`

Notebook делает:
1. создание energy-response признаков;
2. residual hardness;
3. `hardness_score`;
4. несколько способов выделения энергоёмкостных режимов:
   - KMeans;
   - GMM;
   - quantile segmentation по `hardness_score_smooth`;
   - rule-based физическая классификация;
   - segment-level quantile segmentation;
5. интерактивные Plotly 3D maps/surfaces `pressure_axis × pressure_rotation → speed`;
6. сохранение датасета с признаками и всеми вариантами разметки.

Очистка датасета убрана: предполагается, что `united.csv` уже очищен.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

try:
    import joblib
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

RANDOM_STATE = 42
EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Загрузка данных без очистки

In [3]:
DATA_PATH = "../datasets/united.csv"

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Не хватает колонок: {missing}")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed"]].describe(percentiles=[.01, .05, .5, .95, .99]))

Loaded: ../datasets/united.csv
Shape: (415049, 6)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515


,pressure_axis,pressure_rotation,rotation,speed
count,415049.000000,415049.000000,415049.000000,415049.000000
mean,17473.667273,14134.146325,103.945315,0.013116
std,4682.982816,3243.523440,13.370361,0.006615
min,317.000000,784.000000,50.010000,0.001002
1%,3713.000000,6271.000000,64.980000,0.002755
5%,6645.000000,8246.000000,81.750000,0.005050
50%,18861.000000,14637.000000,103.158000,0.012120
95%,22343.000000,18758.600000,138.474000,0.024240
99%,23626.000000,20881.000000,139.020000,0.030300
max,24872.000000,26318.000000,139.578000,0.038957


## 2. Временной шаг

In [4]:
df["dt"] = (
    df.groupby("well_id")["processing_time"]
      .diff()
      .dt.total_seconds()
)

df["dt"] = df["dt"].fillna(df["dt"].median())
display(df["dt"].describe(percentiles=[.01, .05, .5, .95, .99]))

count    415049.000000
mean          7.270602
std         110.393457
min           0.089000
1%            0.229000
5%            0.464000
50%           5.135000
95%          10.891000
99%          23.689560
max       42583.603000
Name: dt, dtype: float64

## 3. Energy-response признаки

In [5]:
df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]

df["pressure_balance"] = (
    df["pressure_axis"] /
    (df["pressure_axis"] + df["pressure_rotation"] + EPS)
)

df["axis_over_rot_pressure"] = (
    df["pressure_axis"] /
    (df["pressure_rotation"] + EPS)
)

df["rot_pressure_over_axis"] = (
    df["pressure_rotation"] /
    (df["pressure_axis"] + EPS)
)

df["rotation_efficiency"] = (
    df["rotation"] /
    (df["pressure_rotation"] + EPS)
)

df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]

df["energy_input_proxy"] = (
    df["pressure_axis"] +
    df["pressure_rotation"] * df["rotation"]
)

df["pseudo_mse"] = (
    df["energy_input_proxy"] /
    (df["speed"] + EPS)
)

df["drilling_efficiency"] = (
    df["speed"] /
    (df["energy_input_proxy"] + EPS)
)

df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"])

display(df[[
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "rotation_efficiency",
    "pressure_balance",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

,energy_input_proxy,pseudo_mse,drilling_efficiency,rotation_efficiency,pressure_balance
count,4.150490e+05,4.150490e+05,4.150490e+05,415049.000000,415049.000000
mean,1.494500e+06,1.467727e+08,9.279041e-09,0.007803,0.545942
std,4.009753e+05,9.155477e+07,5.931875e-09,0.002483,0.057506
min,4.174649e+04,3.009300e+06,3.870720e-10,0.002104,0.013293
1%,4.697380e+05,3.257078e+07,2.153750e-09,0.004789,0.320485
5%,7.635049e+05,5.221720e+07,3.292781e-09,0.005407,0.428681
50%,1.542361e+06,1.239899e+08,8.064516e-09,0.007111,0.558077
95%,2.119089e+06,3.036335e+08,1.914947e-08,0.012153,0.607300
99%,2.462635e+06,4.641533e+08,3.070111e-08,0.015835,0.629086
max,3.668551e+06,2.581497e+09,3.322895e-07,0.121481,0.933514


## 4. Rolling-признаки energy-response

In [6]:
def add_rolling_stats(data, cols, windows=(12, 30, 60), group_col="well_id"):
    out = data.copy()

    for col in cols:
        for w in windows:
            min_p = max(3, w // 3)

            out[f"{col}_roll_median_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).median())
            )

            out[f"{col}_roll_mean_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).mean())
            )

            out[f"{col}_roll_std_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).std())
            )

    return out

rolling_cols = [
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "rotation_efficiency",
    "speed",
    "rotation",
    "pressure_balance",
]

df = add_rolling_stats(df, rolling_cols, windows=(12, 30, 60))

print("Rolling features added:", len([c for c in df.columns if "_roll_" in c]))

Rolling features added: 63


## 5. Residual hardness

In [7]:
control_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "total_pressure",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "axis_x_rotation",
    "rot_pressure_x_rotation",
    "energy_input_proxy",
    "log_energy_input_proxy",
]

residual_df = df.dropna(subset=control_features + ["speed", "well_id"]).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(residual_df, groups=residual_df["well_id"]))

train_res = residual_df.iloc[train_idx].copy()
test_res = residual_df.iloc[test_idx].copy()

expected_speed_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ))
])

expected_speed_model.fit(train_res[control_features], train_res["speed"])

pred_test = expected_speed_model.predict(test_res[control_features])

mae = mean_absolute_error(test_res["speed"], pred_test)
rmse = root_mean_squared_error(test_res["speed"], pred_test)
r2 = r2_score(test_res["speed"], pred_test)

print(f"Expected-speed controls model | MAE={mae:.6f} | RMSE={rmse:.6f} | R2={r2:.4f}")

df["expected_speed_from_controls"] = expected_speed_model.predict(df[control_features])
df["formation_residual"] = df["speed"] - df["expected_speed_from_controls"]
df["relative_formation_residual"] = (
    df["formation_residual"] /
    (np.abs(df["expected_speed_from_controls"]) + EPS)
)

display(df[[
    "speed",
    "expected_speed_from_controls",
    "formation_residual",
    "relative_formation_residual",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

Expected-speed controls model | MAE=0.004750 | RMSE=0.006028 | R2=0.1725


,speed,expected_speed_from_controls,formation_residual,relative_formation_residual
count,415049.000000,415049.000000,4.150490e+05,415049.000000
mean,0.013116,0.013116,3.750782e-07,-0.005738
std,0.006615,0.002594,5.960673e-03,0.459814
min,0.001002,0.005363,-1.699507e-02,-0.936298
1%,0.002755,0.006579,-1.079520e-02,-0.777480
5%,0.005050,0.008102,-8.642517e-03,-0.618527
50%,0.012120,0.013441,-7.497199e-04,-0.061736
95%,0.024240,0.016886,1.057625e-02,0.811159
99%,0.030300,0.017621,1.765780e-02,1.341649
max,0.038957,0.018344,3.055098e-02,5.258323


## 6. Rolling residual-признаки

In [8]:
df = add_rolling_stats(
    df,
    cols=["formation_residual", "relative_formation_residual", "expected_speed_from_controls"],
    windows=(12, 30, 60),
)

display(df[[
    "formation_residual_roll_median_60",
    "relative_formation_residual_roll_median_60",
    "expected_speed_from_controls_roll_median_60",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

,formation_residual_roll_median_60,relative_formation_residual_roll_median_60,expected_speed_from_controls_roll_median_60
count,382635.000000,382635.000000,382635.000000
mean,-0.000321,-0.040641,0.013104
std,0.003772,0.295794,0.001967
min,-0.011658,-0.781768,0.005734
1%,-0.007719,-0.589639,0.007530
5%,-0.005349,-0.476921,0.009496
50%,-0.001026,-0.083054,0.013349
95%,0.006899,0.496963,0.015960
99%,0.010600,0.815860,0.016886
max,0.020247,1.943494,0.017677


## 7. Continuous hardness_score

In [9]:
def zscore(s):
    return (s - s.mean()) / (s.std() + EPS)

df["hardness_score"] = (
    zscore(df["log_pseudo_mse"])
    - zscore(df["drilling_efficiency"])
    - zscore(df["formation_residual"])
)

df["hardness_score_smooth"] = (
    zscore(np.log1p(df["pseudo_mse_roll_median_60"]))
    - zscore(df["drilling_efficiency_roll_median_60"])
    - zscore(df["formation_residual_roll_median_60"])
)

display(df[["hardness_score", "hardness_score_smooth"]].describe(percentiles=[.01, .05, .5, .95, .99]))

,hardness_score,hardness_score_smooth
count,4.150490e+05,3.826350e+05
mean,-4.329998e-15,4.278465e-15
std,1.929089e+00,1.898597e+00
min,-1.016414e+01,-1.023758e+01
1%,-5.092313e+00,-5.313689e+00
5%,-3.279611e+00,-3.437390e+00
50%,8.414619e-02,2.413973e-01
95%,2.963625e+00,2.752807e+00
99%,3.862814e+00,3.826881e+00
max,8.309555e+00,6.586316e+00


## 8. Признаки для KMeans/GMM

In [10]:
cluster_features = [
    "log_pseudo_mse",
    "drilling_efficiency",
    "formation_residual",
    "relative_formation_residual",
    "rotation_efficiency",
    "hardness_score",
    "pseudo_mse_roll_median_60",
    "drilling_efficiency_roll_median_60",
    "formation_residual_roll_median_60",
    "relative_formation_residual_roll_median_60",
    "speed_roll_std_60",
    "rotation_roll_std_60",
    "hardness_score_smooth",
]

cluster_df = df.dropna(subset=cluster_features + ["well_id", "processing_time"]).copy()

cluster_df["log_pseudo_mse_roll_median_60"] = np.log1p(cluster_df["pseudo_mse_roll_median_60"])
cluster_df["log_speed_roll_std_60"] = np.log1p(cluster_df["speed_roll_std_60"])
cluster_df["log_rotation_roll_std_60"] = np.log1p(cluster_df["rotation_roll_std_60"])

cluster_features_final = [
    "log_pseudo_mse",
    "drilling_efficiency",
    "formation_residual",
    "relative_formation_residual",
    "rotation_efficiency",
    "hardness_score",
    "log_pseudo_mse_roll_median_60",
    "drilling_efficiency_roll_median_60",
    "formation_residual_roll_median_60",
    "relative_formation_residual_roll_median_60",
    "log_speed_roll_std_60",
    "log_rotation_roll_std_60",
    "hardness_score_smooth",
]

cluster_scaler = StandardScaler()
X = cluster_scaler.fit_transform(cluster_df[cluster_features_final])

print("cluster_df:", cluster_df.shape)
display(cluster_df[cluster_features_final].describe(percentiles=[.01, .05, .5, .95, .99]))

cluster_df: (382635, 117)


,log_pseudo_mse,drilling_efficiency,formation_residual,relative_formation_residual,rotation_efficiency,hardness_score,log_pseudo_mse_roll_median_60,drilling_efficiency_roll_median_60,formation_residual_roll_median_60,relative_formation_residual_roll_median_60,log_speed_roll_std_60,log_rotation_roll_std_60,hardness_score_smooth
count,382635.000000,3.826350e+05,382635.000000,382635.000000,382635.000000,382635.000000,382635.000000,3.826350e+05,382635.000000,382635.000000,382635.000000,382635.000000,3.826350e+05
mean,18.683644,8.770042e-09,-0.000078,-0.011561,0.007539,0.074797,18.608537,8.994513e-09,-0.000321,-0.040641,0.004870,1.532169,4.316496e-15
std,0.520949,4.791997e-09,0.005785,0.441195,0.002174,1.853509,0.394298,4.053775e-09,0.003772,0.295794,0.001417,0.968986,1.898597e+00
min,14.917218,3.870720e-10,-0.016995,-0.936298,0.002104,-10.164138,16.229518,1.909207e-09,-0.011658,-0.781768,0.001095,0.159768,-1.023758e+01
1%,17.517959,2.173566e-09,-0.010794,-0.760974,0.004770,-4.651458,17.544988,3.597721e-09,-0.007719,-0.589639,0.002330,0.454677,-5.313689e+00
5%,17.878410,3.288626e-09,-0.008624,-0.612258,0.005380,-3.036935,17.923313,4.306601e-09,-0.005349,-0.476921,0.002983,0.537787,-3.437390e+00
50%,18.656817,7.896054e-09,-0.000771,-0.063347,0.006981,0.132977,18.643845,8.000831e-09,-0.001026,-0.083054,0.004601,1.013815,2.413973e-01
95%,19.532587,1.719816e-08,0.010141,0.771055,0.011374,2.967968,19.263190,1.644376e-08,0.006899,0.496963,0.007592,3.019070,2.752807e+00
99%,19.946567,2.466183e-08,0.016560,1.250706,0.014265,3.834306,19.442847,2.401474e-08,0.010600,0.815860,0.009069,3.381142,3.826881e+00
max,21.671635,3.322895e-07,0.030551,5.258323,0.121481,8.309555,20.076924,8.964683e-08,0.020247,1.943494,0.012966,3.743559,6.586316e+00


## 9. KMeans и GMM кластеризация

In [11]:
K = 4

kmeans_model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init="auto")
cluster_df["label_kmeans"] = kmeans_model.fit_predict(X)

gmm_model = GaussianMixture(
    n_components=K,
    covariance_type="full",
    random_state=RANDOM_STATE,
    reg_covar=1e-6,
)
cluster_df["label_gmm"] = gmm_model.fit_predict(X)


def add_ranked_energy_names(data, label_col, output_col):
    out = data.copy()

    order = (
        out.groupby(label_col)["hardness_score_smooth"]
           .median()
           .sort_values()
           .index
           .tolist()
    )

    if len(order) == 4:
        names = ["soft_low_energy", "medium_low_energy", "medium_high_energy", "hard_high_energy"]
    elif len(order) == 3:
        names = ["soft_low_energy", "medium_energy", "hard_high_energy"]
    else:
        names = [f"energy_rank_{i}" for i in range(len(order))]

    mapping = {cl: names[i] for i, cl in enumerate(order)}
    out[output_col] = out[label_col].map(mapping)
    return out


cluster_df = add_ranked_energy_names(cluster_df, "label_kmeans", "energy_type_kmeans")
cluster_df = add_ranked_energy_names(cluster_df, "label_gmm", "energy_type_gmm")

display(cluster_df[["label_kmeans", "energy_type_kmeans", "label_gmm", "energy_type_gmm"]].head())

,label_kmeans,energy_type_kmeans,label_gmm,energy_type_gmm
19,2,soft_low_energy,2,soft_low_energy
20,2,soft_low_energy,2,soft_low_energy
21,2,soft_low_energy,2,soft_low_energy
22,2,soft_low_energy,2,soft_low_energy
23,2,soft_low_energy,2,soft_low_energy


## 10. Alternative segmentation 1: quantile по hardness_score_smooth

In [12]:
energy_labels_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

cluster_df["energy_type_quantile"] = pd.qcut(
    cluster_df["hardness_score_smooth"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

display(
    cluster_df
    .groupby("energy_type_quantile")
    .agg(
        rows=("speed", "size"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
        efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
        residual_smooth=("formation_residual_roll_median_60", "median"),
        speed_median=("speed", "median"),
    )
    .sort_values("hardness_smooth")
)

,rows,hardness_smooth,pseudo_mse_smooth,efficiency_smooth,residual_smooth,speed_median
energy_type_quantile,,,,,,
soft_low_energy,95659,-2.212806,7.893292e+07,1.266945e-08,0.004005,0.01818
medium_low_energy,95661,-0.352565,1.115156e+08,8.968309e-09,0.000256,0.01212
medium_high_energy,95656,0.736440,1.371730e+08,7.290481e-09,-0.001889,0.01212
hard_high_energy,95659,2.035933,1.898602e+08,5.274633e-09,-0.003889,0.00606


## 11. Alternative segmentation 2: rule-based физическая классификация

In [14]:
# Пороговые значения по физическим proxy
q_pseudo_25 = cluster_df["pseudo_mse_roll_median_60"].quantile(0.25)
q_pseudo_75 = cluster_df["pseudo_mse_roll_median_60"].quantile(0.75)

q_eff_25 = cluster_df["drilling_efficiency_roll_median_60"].quantile(0.25)
q_eff_75 = cluster_df["drilling_efficiency_roll_median_60"].quantile(0.75)

q_res_25 = cluster_df["formation_residual_roll_median_60"].quantile(0.25)
q_res_75 = cluster_df["formation_residual_roll_median_60"].quantile(0.75)

hard_votes = (
    (cluster_df["pseudo_mse_roll_median_60"] >= q_pseudo_75).astype(np.int8)
    + (cluster_df["drilling_efficiency_roll_median_60"] <= q_eff_25).astype(np.int8)
    + (cluster_df["formation_residual_roll_median_60"] <= q_res_25).astype(np.int8)
)

soft_votes = (
    (cluster_df["pseudo_mse_roll_median_60"] <= q_pseudo_25).astype(np.int8)
    + (cluster_df["drilling_efficiency_roll_median_60"] >= q_eff_75).astype(np.int8)
    + (cluster_df["formation_residual_roll_median_60"] >= q_res_75).astype(np.int8)
)

hardness_median = cluster_df["hardness_score_smooth"].median()
middle_labels = np.where(
    cluster_df["hardness_score_smooth"] <= hardness_median,
    "medium_low_energy",
    "medium_high_energy",
)

cluster_df["energy_type_rule_based"] = np.select(
    [hard_votes >= 2, soft_votes >= 2],
    ["hard_high_energy", "soft_low_energy"],
    default=middle_labels,
)

display(
    cluster_df
    .groupby("energy_type_rule_based")
    .agg(
        rows=("speed", "size"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
        efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
        residual_smooth=("formation_residual_roll_median_60", "median"),
        speed_median=("speed", "median"),
    )
    .sort_values("hardness_smooth")
)

,rows,hardness_smooth,pseudo_mse_smooth,efficiency_smooth,residual_smooth,speed_median
energy_type_rule_based,,,,,,
soft_low_energy,95653,-2.205046,7.791739e+07,1.283523e-08,0.003777,0.018180
medium_low_energy,96202,-0.412886,1.114920e+08,8.969926e-09,0.000781,0.013773
medium_high_energy,95098,0.746105,1.364802e+08,7.327845e-09,-0.002104,0.012120
hard_high_energy,95682,2.032597,1.898602e+08,5.274633e-09,-0.003589,0.006060


## 12. Alternative segmentation 3: segment-level quantile

In [15]:
# Сегментируем внутри well_id по фиксированному числу точек
SEGMENT_SIZE = 60

cluster_df = cluster_df.sort_values(["well_id", "processing_time"]).copy()
cluster_df["_row_in_well"] = cluster_df.groupby("well_id").cumcount()
cluster_df["segment_id"] = (cluster_df["_row_in_well"] // SEGMENT_SIZE).astype(int)

segment_df = (
    cluster_df
    .groupby(["well_id", "segment_id"])
    .agg(
        segment_start=("processing_time", "min"),
        segment_end=("processing_time", "max"),
        rows=("speed", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_60", "median"),
        efficiency_segment=("drilling_efficiency_roll_median_60", "median"),
        residual_segment=("formation_residual_roll_median_60", "median"),
        speed_segment=("speed", "median"),
    )
    .reset_index()
)

segment_df["energy_type_segment_quantile"] = pd.qcut(
    segment_df["hardness_segment"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

cluster_df = cluster_df.merge(
    segment_df[["well_id", "segment_id", "energy_type_segment_quantile", "hardness_segment"]],
    on=["well_id", "segment_id"],
    how="left",
)

display(
    segment_df
    .groupby("energy_type_segment_quantile")
    .agg(
        segments=("segment_id", "size"),
        rows=("rows", "sum"),
        hardness_segment=("hardness_segment", "median"),
        pseudo_mse_segment=("pseudo_mse_segment", "median"),
        efficiency_segment=("efficiency_segment", "median"),
        residual_segment=("residual_segment", "median"),
        speed_segment=("speed_segment", "median"),
    )
    .sort_values("hardness_segment")
)

,segments,rows,hardness_segment,pseudo_mse_segment,efficiency_segment,residual_segment,speed_segment
energy_type_segment_quantile,,,,,,,
soft_low_energy,1801,94049,-2.186504,7.979763e+07,1.253212e-08,0.004020,0.01818
medium_low_energy,1800,96213,-0.362486,1.118751e+08,8.938281e-09,0.000371,0.01212
medium_high_energy,1800,95943,0.716003,1.369420e+08,7.304144e-09,-0.001799,0.01212
hard_high_energy,1801,96430,1.979057,1.865598e+08,5.363636e-09,-0.003793,0.00606


## 13. Сводка всех методов

In [16]:
def summarize_energy_clusters(data, type_col):
    return (
        data
        .groupby(type_col)
        .agg(
            rows=("speed", "size"),
            speed_median=("speed", "median"),
            pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
            efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
            residual_smooth=("formation_residual_roll_median_60", "median"),
            hardness_smooth=("hardness_score_smooth", "median"),
            pressure_axis_median=("pressure_axis", "median"),
            pressure_rotation_median=("pressure_rotation", "median"),
            rotation_median=("rotation", "median"),
            energy_input_median=("energy_input_proxy", "median"),
        )
        .sort_values("hardness_smooth")
    )

SEGMENTATION_METHODS = [
    "energy_type_kmeans",
    "energy_type_gmm",
    "energy_type_quantile",
    "energy_type_rule_based",
    "energy_type_segment_quantile",
]

for method in SEGMENTATION_METHODS:
    print("\n" + "=" * 80)
    print(method)
    display(summarize_energy_clusters(cluster_df, method))


energy_type_kmeans


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median,energy_input_median
energy_type_kmeans,,,,,,,,,,
soft_low_energy,58853,0.02121,7.297385e+07,1.370572e-08,0.004854,-2.672623,17195.0,14160.0,103.308,1488004.342
medium_low_energy,70095,0.01212,9.538767e+07,1.048473e-08,0.001632,-1.094870,18062.0,14748.0,103.602,1573057.740
medium_high_energy,129318,0.01515,1.288308e+08,7.762442e-09,-0.001131,0.344682,20190.0,15470.0,102.714,1616134.133
hard_high_energy,124369,0.00606,1.611613e+08,6.206590e-09,-0.003151,1.559755,19405.0,14510.0,103.308,1537123.018



energy_type_gmm


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median,energy_input_median
energy_type_gmm,,,,,,,,,,
soft_low_energy,48316,0.01818,7.227633e+07,1.383792e-08,0.001330,-1.881694,15067.0,11526.0,103.704,1194177.379
medium_low_energy,90074,0.01515,8.772975e+07,1.139930e-08,0.002532,-1.561306,17410.0,14334.0,103.308,1514089.719
medium_high_energy,122965,0.01212,1.265284e+08,7.903647e-09,-0.000605,0.184594,20572.0,16115.0,102.816,1684847.888
hard_high_energy,121280,0.00606,1.690452e+08,5.918563e-09,-0.003100,1.684015,19473.0,14356.0,103.458,1521607.128



energy_type_quantile


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median,energy_input_median
energy_type_quantile,,,,,,,,,,
soft_low_energy,95659,0.01818,7.893292e+07,1.266945e-08,0.004005,-2.212806,17842.0,14740.0,103.308,1553204.240
medium_low_energy,95661,0.01212,1.115156e+08,8.968309e-09,0.000256,-0.352565,19628.0,15178.0,103.008,1594642.406
medium_high_energy,95656,0.01212,1.371730e+08,7.290481e-09,-0.001889,0.736440,20085.0,15244.0,102.912,1601603.286
hard_high_energy,95659,0.00606,1.898602e+08,5.274633e-09,-0.003889,2.035933,19187.0,14331.0,103.458,1518641.848



energy_type_rule_based


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median,energy_input_median
energy_type_rule_based,,,,,,,,,,
soft_low_energy,95653,0.018180,7.791739e+07,1.283523e-08,0.003777,-2.205046,16720.0,13605.0,103.410,1430114.622
medium_low_energy,96202,0.013773,1.114920e+08,8.969926e-09,0.000781,-0.412886,20341.0,15720.0,103.008,1657545.544
medium_high_energy,95098,0.012120,1.364802e+08,7.327845e-09,-0.002104,0.746105,19636.0,15160.0,102.864,1589148.153
hard_high_energy,95682,0.006060,1.898602e+08,5.274633e-09,-0.003589,2.032597,19682.0,14484.0,103.458,1537272.999



energy_type_segment_quantile


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median,energy_input_median
energy_type_segment_quantile,,,,,,,,,,
soft_low_energy,94049,0.01818,7.974684e+07,1.254017e-08,0.003969,-2.194668,17867.0,14759.0,103.308,1554756.032
medium_low_energy,96213,0.01212,1.116768e+08,8.954801e-09,0.000290,-0.354254,19606.0,15166.0,103.008,1593824.100
medium_high_energy,95943,0.01212,1.369806e+08,7.300751e-09,-0.001792,0.714666,20037.0,15180.0,102.966,1595171.720
hard_high_energy,96430,0.00606,1.858249e+08,5.388009e-09,-0.003750,1.983451,19231.0,14404.0,103.410,1525925.468


## 14. Plotly 3D карта фактических точек

In [17]:
PLOT_SAMPLE = min(50000, len(cluster_df))
plot_df = cluster_df.sample(PLOT_SAMPLE, random_state=RANDOM_STATE).copy()

COLOR_BY = "energy_type_segment_quantile"  # можно заменить на любой метод из SEGMENTATION_METHODS

fig = px.scatter_3d(
    plot_df,
    x="pressure_axis",
    y="pressure_rotation",
    z="speed",
    color=COLOR_BY,
    opacity=0.55,
    title=f"Observed 3D map: pressure_axis × pressure_rotation → speed | color={COLOR_BY}",
)

fig.update_traces(marker=dict(size=2))
fig.update_layout(height=750)
fig.show()

## 15. Plotly 3D surface по выбранному типу и методу

In [18]:
SURFACE_TYPE_COL = "energy_type_segment_quantile"  # можно заменить
SURFACE_TYPE = "hard_high_energy"

surface_train = cluster_df[cluster_df[SURFACE_TYPE_COL] == SURFACE_TYPE].dropna(subset=[
    "pressure_axis", "pressure_rotation", "rotation", "hardness_score_smooth", "speed"
]).copy()

print("Surface type col:", SURFACE_TYPE_COL)
print("Surface type:", SURFACE_TYPE)
print("Rows:", len(surface_train))

surface_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "hardness_score_smooth",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "energy_input_proxy",
]

surface_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ))
])

surface_model.fit(surface_train[surface_features], surface_train["speed"])

fixed_state = surface_train[surface_features].median()

p_ax_grid = np.linspace(
    surface_train["pressure_axis"].quantile(0.05),
    surface_train["pressure_axis"].quantile(0.95),
    60,
)

p_rot_grid = np.linspace(
    surface_train["pressure_rotation"].quantile(0.05),
    surface_train["pressure_rotation"].quantile(0.95),
    60,
)

PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

grid = pd.DataFrame({
    "pressure_axis": PA.ravel(),
    "pressure_rotation": PR.ravel(),
})

for col in surface_features:
    if col not in grid.columns:
        grid[col] = fixed_state[col]

grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

Z = surface_model.predict(grid[surface_features]).reshape(PA.shape)

fig = go.Figure()

fig.add_trace(go.Surface(
    x=PA,
    y=PR,
    z=Z,
    colorscale="Viridis",
    opacity=0.85,
    name="predicted surface",
))

sample_points = surface_train.sample(min(3000, len(surface_train)), random_state=RANDOM_STATE)

fig.add_trace(go.Scatter3d(
    x=sample_points["pressure_axis"],
    y=sample_points["pressure_rotation"],
    z=sample_points["speed"],
    mode="markers",
    marker=dict(size=2, opacity=0.35),
    name="observed points",
))

fig.update_layout(
    title=f"Interactive surface: p_ax × p_rot → speed | {SURFACE_TYPE_COL}={SURFACE_TYPE}",
    scene=dict(
        xaxis_title="pressure_axis",
        yaxis_title="pressure_rotation",
        zaxis_title="speed",
    ),
    height=800,
)

fig.show()

Surface type col: energy_type_segment_quantile
Surface type: hard_high_energy
Rows: 96430


## 16. Plotly surfaces для всех классов выбранного метода

In [19]:
for surface_type in sorted(cluster_df[SURFACE_TYPE_COL].dropna().unique()):
    surface_train = cluster_df[cluster_df[SURFACE_TYPE_COL] == surface_type].dropna(subset=[
        "pressure_axis", "pressure_rotation", "rotation", "hardness_score_smooth", "speed"
    ]).copy()

    if len(surface_train) < 500:
        print("Skip small class:", surface_type, len(surface_train))
        continue

    surface_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        ))
    ])

    surface_model.fit(surface_train[surface_features], surface_train["speed"])
    fixed_state = surface_train[surface_features].median()

    p_ax_grid = np.linspace(surface_train["pressure_axis"].quantile(0.05), surface_train["pressure_axis"].quantile(0.95), 50)
    p_rot_grid = np.linspace(surface_train["pressure_rotation"].quantile(0.05), surface_train["pressure_rotation"].quantile(0.95), 50)

    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({"pressure_axis": PA.ravel(), "pressure_rotation": PR.ravel()})

    for col in surface_features:
        if col not in grid.columns:
            grid[col] = fixed_state[col]

    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

    Z = surface_model.predict(grid[surface_features]).reshape(PA.shape)

    fig = go.Figure()
    fig.add_trace(go.Surface(x=PA, y=PR, z=Z, colorscale="Viridis", opacity=0.85))

    sample_points = surface_train.sample(min(2000, len(surface_train)), random_state=RANDOM_STATE)

    fig.add_trace(go.Scatter3d(
        x=sample_points["pressure_axis"],
        y=sample_points["pressure_rotation"],
        z=sample_points["speed"],
        mode="markers",
        marker=dict(size=2, opacity=0.25),
        name="observed points",
    ))

    fig.update_layout(
        title=f"Surface by {SURFACE_TYPE_COL}: {surface_type}",
        scene=dict(
            xaxis_title="pressure_axis",
            yaxis_title="pressure_rotation",
            zaxis_title="speed",
        ),
        height=750,
    )

    fig.show()

## 17. Возвращаем разметку в основной датасет

In [20]:
cols_to_merge = [
    "processing_time",
    "well_id",
    "label_kmeans",
    "energy_type_kmeans",
    "label_gmm",
    "energy_type_gmm",
    "energy_type_quantile",
    "energy_type_rule_based",
    "segment_id",
    "hardness_segment",
    "energy_type_segment_quantile",
    "hardness_score",
    "hardness_score_smooth",
]

df_out = df.merge(
    cluster_df[cols_to_merge],
    on=["processing_time", "well_id"],
    how="left",
    suffixes=("", "_segmented"),
)

FINAL_METHOD = "segment_quantile"

if FINAL_METHOD == "gmm":
    df_out["rock_energy_label_final"] = df_out["label_gmm"]
    df_out["rock_energy_type_final"] = df_out["energy_type_gmm"]
elif FINAL_METHOD == "kmeans":
    df_out["rock_energy_label_final"] = df_out["label_kmeans"]
    df_out["rock_energy_type_final"] = df_out["energy_type_kmeans"]
elif FINAL_METHOD == "quantile":
    df_out["rock_energy_label_final"] = np.nan
    df_out["rock_energy_type_final"] = df_out["energy_type_quantile"]
elif FINAL_METHOD == "rule_based":
    df_out["rock_energy_label_final"] = np.nan
    df_out["rock_energy_type_final"] = df_out["energy_type_rule_based"]
else:
    df_out["rock_energy_label_final"] = np.nan
    df_out["rock_energy_type_final"] = df_out["energy_type_segment_quantile"]

display(df_out[[
    "processing_time",
    "well_id",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "formation_residual",
    "hardness_score_smooth",
    "energy_type_kmeans",
    "energy_type_gmm",
    "energy_type_quantile",
    "energy_type_rule_based",
    "energy_type_segment_quantile",
    "rock_energy_type_final",
]].head())

,processing_time,well_id,speed,energy_input_proxy,pseudo_mse,drilling_efficiency,formation_residual,hardness_score_smooth,energy_type_kmeans,energy_type_gmm,energy_type_quantile,energy_type_rule_based,energy_type_segment_quantile,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,0.002755,338743.056,1.229314e+08,8.131666e-09,-0.005300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,19601,0.003030,279919.984,9.235235e+07,1.082452e-08,-0.005025,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-08-24 10:10:05.199,19601,0.006060,324846.384,5.359617e+07,1.865497e-08,-0.001995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-08-24 10:10:14.610,19601,0.002755,274750.210,9.970810e+07,1.002564e-08,-0.005300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-08-24 10:10:34.197,19601,0.001515,289195.048,1.907619e+08,5.238679e-09,-0.006540,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 18. Сохранение результата

In [21]:
OUTPUT_PATH = "united_rock_energy_features_all_methods.csv"
CONFIG_PATH = "rock_energy_feature_clustering_all_methods_config.json"

df_out.to_csv(OUTPUT_PATH, index=False)

config = {
    "data_path": DATA_PATH,
    "output_path": OUTPUT_PATH,
    "final_method": FINAL_METHOD,
    "segmentation_methods": [
        "energy_type_kmeans",
        "energy_type_gmm",
        "energy_type_quantile",
        "energy_type_rule_based",
        "energy_type_segment_quantile",
    ],
    "control_features_for_residual": control_features,
    "cluster_features_final": cluster_features_final,
    "surface_features": surface_features,
    "segment_size": SEGMENT_SIZE,
    "labels": {
        "kmeans": "energy_type_kmeans",
        "gmm": "energy_type_gmm",
        "quantile": "energy_type_quantile",
        "rule_based": "energy_type_rule_based",
        "segment_quantile": "energy_type_segment_quantile",
        "final": "rock_energy_type_final",
    },
    "interpretation": {
        "hardness_score_smooth": "continuous operational drilling resistance index",
        "rock_energy_type_final": "discrete energy-response regime, not true lithology",
        "pseudo_mse": "proxy energy per penetration, not physical MSE",
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Saved:", CONFIG_PATH)

if HAS_JOBLIB:
    ARTIFACT_DIR = Path("rock_energy_all_methods_artifacts")
    ARTIFACT_DIR.mkdir(exist_ok=True)

    joblib.dump(expected_speed_model, ARTIFACT_DIR / "expected_speed_from_controls_model.joblib")
    joblib.dump(cluster_scaler, ARTIFACT_DIR / "energy_cluster_scaler.joblib")
    joblib.dump(kmeans_model, ARTIFACT_DIR / "energy_kmeans_model.joblib")
    joblib.dump(gmm_model, ARTIFACT_DIR / "energy_gmm_model.joblib")

    print("Saved artifacts to:", ARTIFACT_DIR)

Saved: united_rock_energy_features_all_methods.csv
Saved: rock_energy_feature_clustering_all_methods_config.json
Saved artifacts to: rock_energy_all_methods_artifacts
